In [1]:
import pandas as pd
import glob
import numpy as np
from datetime import time

In [ ]:
import glob
import pandas as pd
from datetime import time

START_TIME = time(9,17)
EXIT_TIME = time(15,16)
REENTRY_END = time(9,34)

OFF_SET_TH = 0.2
QTY = 1

MAX_TRADES_CE = 2
MAX_TRADES_PE = 2

FIXED_SL_ACTIVE_TILL = time(9,34)

TRAIL_START = 0.5
TRAIL_RATIO = 0.5
TRAIL_ACTIVE_TILL = time(9,34)

ATP_TRAIL_START = time(9,35)
ATP_MULT = 1.10

result = []

files = glob.glob('data/nifty*.parquet')
files = files[-3:]


for f in sorted(files):

  df = pd.read_parquet(f)
  df = df.sort_values('datetime')

  pivot = (
      df.pivot_table(
          index=['datetime','strike_price'],
          columns='right',
          values='close_opt',
          aggfunc='last'
      )
      .reset_index()
      .dropna(subset=['Call','Put'])
  )

  pivot["straddle"] = pivot["Call"] + pivot["Put"]

  expiry = pd.to_datetime(df['expiry'].iloc[0]).strftime('%d%b%y')
  dte = (pd.to_datetime(df['expiry'].iloc[0]) - pd.to_datetime(df['date'].iloc[0])).days

  open_ce_trades = []
  open_pe_trades = []

  for i,row in df.groupby("datetime"):

      if i.time() < START_TIME:
          continue

      spot_close = row['close_spot'].iloc[0]
      atm_strike = round(spot_close/50)*50

      straddle_row = pivot[(pivot['datetime']==i) & (pivot['strike_price']==atm_strike)]

      if straddle_row.empty:
          continue

      straddle_p = straddle_row['straddle'].iloc[0]
      off_set = OFF_SET_TH * straddle_p

      call_row = row[
          (row["right"]=="Call") &
          (row["strike_price"] > atm_strike-50) &
          (row["close_opt"] <= off_set)
      ].sort_values("strike_price").head(1)

      put_row = row[
          (row["right"]=="Put") &
          (row["strike_price"] < atm_strike+50) &
          (row["close_opt"] <= off_set)
      ].sort_values("strike_price",ascending=False).head(1)

      if call_row.empty or put_row.empty:
          continue

      call_strike = call_row["strike_price"].iloc[0]
      put_strike = put_row["strike_price"].iloc[0]

      call_close = call_row["close_opt"].iloc[0]
      put_close = put_row["close_opt"].iloc[0]

      call_atp = df[(df['strike_price']==call_strike)&(df['datetime']<=i)&(df['right']=='Call')]['close_opt'].mean()
      put_atp = df[(df['strike_price']==put_strike)&(df['datetime']<=i)&(df['right']=='Put')]['close_opt'].mean()


      


      if i.time() <= REENTRY_END:

          if not open_ce_trades and call_close < call_atp:

              open_ce_trades.append({
                  "entry_time": i,
                  "entry_price": call_close,
                  "strike": call_strike,
                  "sl": call_close*1.5,
                  "trail_sl": call_close*1.5,
                  "spot": spot_close
              })

          if not open_pe_trades and put_close < put_atp:

              open_pe_trades.append({
                  "entry_time": i,
                  "entry_price": put_close,
                  "strike": put_strike,
                  "sl": put_close*1.5,
                  "trail_sl": put_close*1.5,
                  "spot": spot_close
              })


      


      for trade in open_ce_trades.copy():

          ce_ltp = row[(row['strike_price']==trade["strike"])&(row['right']=='Call')]['close_opt'].iloc[0]
          ce_high = row[(row['strike_price']==trade["strike"])&(row['right']=='Call')]['high_opt'].iloc[0]

          profit_move = trade["entry_price"] - ce_ltp

          trail_hit = False

          if i.time() <= TRAIL_ACTIVE_TILL and profit_move >= trade["entry_price"]*TRAIL_START:

              extra_profit = profit_move - (trade["entry_price"]*TRAIL_START)
              trail_reduction = extra_profit * TRAIL_RATIO

              new_trail_sl = trade["sl"] - trail_reduction
              trade["trail_sl"] = min(trade["trail_sl"], new_trail_sl)

              trail_hit = ce_ltp >= trade["trail_sl"]

          sl_hit = ce_ltp >= trade["sl"]

          fixed_sl_hit = False
          if i.time() <= FIXED_SL_ACTIVE_TILL:
              fixed_sl_hit = ce_high >= trade["sl"]

          atp_trail_hit = False
          if i.time() >= ATP_TRAIL_START:
              atp_trail_hit = ce_high >= call_atp * ATP_MULT

          time_exit = i.time() >= EXIT_TIME

          if sl_hit or fixed_sl_hit or trail_hit or atp_trail_hit or time_exit:

              if fixed_sl_hit:
                  exit_price = trade["sl"]
                  reason = 'fixed SL hit (50%)'

              elif atp_trail_hit:
                  exit_price = call_atp * ATP_MULT
                  reason = 'ATP trailing hit'

              elif trail_hit:
                  exit_price = trade["trail_sl"]
                  reason = 'Profit trailing SL hit'

              elif sl_hit:
                  exit_price = trade["sl"]
                  reason = 'Initial SL hit'

              else:
                  exit_price = ce_ltp
                  reason = 'time_exit'

              result.append({
                  'trade_status': 'close',
                  'instrument_id': 'NIFTY',
                  'symbol': f'NIFTY{expiry}{trade["strike"]}CE',
                  'option_type': 'CE',
                  'buy/sell': 'sell',
                  'entry_date': trade["entry_time"].date(),
                  'dte': dte,
                  'expiry': pd.to_datetime(expiry).strftime('%Y-%m-%d'),
                  'entry_signal_time': trade["entry_time"],
                  'entry_time': trade["entry_time"],
                  'entry_price': trade["entry_price"],
                  'entry_reason': 'Call trade occured',
                  'quantity': QTY,
                  'market_bias': None,
                  'stop_loss': trade["sl"],
                  'trailing_sl': call_atp * ATP_MULT,
                  'exit_date': i.strftime('%Y-%m-%d'),
                  'exit_signal_time': i,
                  'exit_price': exit_price,
                  'exit_reason': reason,
                  'pnl_raw': trade["entry_price"] - exit_price,
                  'max_loss': trade["sl"] - trade["entry_price"],
                  'max_profit': trade["entry_price"],
                  'spot_close': trade["spot"],
              })

              open_ce_trades.remove(trade)


              


              if i.time() <= REENTRY_END and len(open_pe_trades) < MAX_TRADES_PE:

                  if put_close < put_atp:

                      open_pe_trades.append({
                          "entry_time": i,
                          "entry_price": put_close,
                          "strike": put_strike,
                          "sl": put_close*1.5,
                          "trail_sl": put_close*1.5,
                          "spot": spot_close
                      })


      


      for trade in open_pe_trades.copy():

          pe_ltp = row[(row['strike_price']==trade["strike"])&(row['right']=='Put')]['close_opt'].iloc[0]
          pe_high = row[(row['strike_price']==trade["strike"])&(row['right']=='Put')]['high_opt'].iloc[0]

          profit_move = trade["entry_price"] - pe_ltp

          trail_hit = False

          if i.time() <= TRAIL_ACTIVE_TILL and profit_move >= trade["entry_price"]*TRAIL_START:

              extra_profit = profit_move - (trade["entry_price"]*TRAIL_START)
              trail_reduction = extra_profit * TRAIL_RATIO

              new_trail_sl = trade["sl"] - trail_reduction
              trade["trail_sl"] = min(trade["trail_sl"], new_trail_sl)

              trail_hit = pe_ltp >= trade["trail_sl"]

          sl_hit = pe_ltp >= trade["sl"]

          fixed_sl_hit = False
          if i.time() <= FIXED_SL_ACTIVE_TILL:
              fixed_sl_hit = pe_high >= trade["sl"]

          atp_trail_hit = False
          if i.time() >= ATP_TRAIL_START:
              atp_trail_hit = pe_high >= put_atp * ATP_MULT

          time_exit = i.time() >= EXIT_TIME

          if sl_hit or fixed_sl_hit or trail_hit or atp_trail_hit or time_exit:

              if fixed_sl_hit:
                  exit_price = trade["sl"]
                  reason = 'fixed SL hit (50%)'

              elif atp_trail_hit:
                  exit_price = put_atp * ATP_MULT
                  reason = 'ATP trailing hit'

              elif trail_hit:
                  exit_price = trade["trail_sl"]
                  reason = 'Profit trailing SL hit'

              elif sl_hit:
                  exit_price = trade["sl"]
                  reason = 'Initial SL hit'

              else:
                  exit_price = pe_ltp
                  reason = 'time_exit'

              result.append({
                  'trade_status': 'close',
                  'instrument_id': 'NIFTY',
                  'symbol': f'NIFTY{expiry}{trade["strike"]}PE',
                  'option_type': 'PE',
                  'buy/sell': 'sell',
                  'entry_date': trade["entry_time"].date(),
                  'dte': dte,
                  'expiry': pd.to_datetime(expiry).strftime('%Y-%m-%d'),
                  'entry_signal_time': trade["entry_time"],
                  'entry_time': trade["entry_time"],
                  'entry_price': trade["entry_price"],
                  'entry_reason': 'Put trade occured',
                  'quantity': QTY,
                  'market_bias': None,
                  'stop_loss': trade["sl"],
                  'trailing_sl': put_atp * ATP_MULT,
                  'exit_date': i.strftime('%Y-%m-%d'),
                  'exit_signal_time': i,
                  'exit_price': exit_price,
                  'exit_reason': reason,
                  'pnl_raw': trade["entry_price"] - exit_price,
                  'max_loss': trade["sl"] - trade["entry_price"],
                  'max_profit': trade["entry_price"],
                  'spot_close': trade["spot"],
              })

              open_pe_trades.remove(trade)


              


              if i.time() <= REENTRY_END and len(open_ce_trades) < MAX_TRADES_CE:

                  if call_close < call_atp:

                      open_ce_trades.append({
                          "entry_time": i,
                          "entry_price": call_close,
                          "strike": call_strike,
                          "sl": call_close*1.5,
                          "trail_sl": call_close*1.5,
                          "spot": spot_close
                      })

results_df = pd.DataFrame(result).sort_values('entry_time')

In [3]:
results_df

,trade_status,instrument_id,symbol,option_type,buy/sell,entry_date,dte,expiry,entry_signal_time,entry_time,...,stop_loss,trailing_sl,exit_date,exit_signal_time,exit_price,exit_reason,pnl_raw,max_loss,max_profit,spot_close
1,close,NIFTY,NIFTY24Feb2625700CE,CE,sell,2026-02-20,4,2026-02-24,2026-02-20 09:17:00,2026-02-20 09:17:00,...,62.775,45.794048,2026-02-20,2026-02-20 09:35:00,45.794048,ATP trailing hit,-3.944048,20.925,41.85,25440.75
0,close,NIFTY,NIFTY24Feb2625200PE,PE,sell,2026-02-20,4,2026-02-24,2026-02-20 09:19:00,2026-02-20 09:19:00,...,67.725,39.237000,2026-02-20,2026-02-20 09:29:00,67.725000,fixed SL hit (50%),-22.575000,22.575,45.15,25452.90
2,close,NIFTY,NIFTY24Feb2625650CE,CE,sell,2026-02-20,4,2026-02-24,2026-02-20 09:29:00,2026-02-20 09:29:00,...,65.175,45.794048,2026-02-20,2026-02-20 09:35:00,45.794048,ATP trailing hit,-2.344048,21.725,43.45,25382.10
3,close,NIFTY,NIFTY24Feb2625200PE,PE,sell,2026-02-20,4,2026-02-24,2026-02-20 09:33:00,2026-02-20 09:33:00,...,70.650,51.116000,2026-02-20,2026-02-20 10:09:00,51.116000,ATP trailing hit,-4.016000,23.550,47.10,25451.95
4,close,NIFTY,NIFTY24Feb2625500PE,PE,sell,2026-02-23,1,2026-02-24,2026-02-23 09:17:00,2026-02-23 09:17:00,...,60.525,38.428077,2026-02-23,2026-02-23 09:53:00,38.428077,ATP trailing hit,1.921923,20.175,40.35,25701.10
5,close,NIFTY,NIFTY24Feb2625450PE,PE,sell,2026-02-24,0,2026-02-24,2026-02-24 09:17:00,2026-02-24 09:17:00,...,28.050,15.413750,2026-02-24,2026-02-24 09:22:00,28.050000,fixed SL hit (50%),-9.350000,9.350,18.70,25593.95
7,close,NIFTY,NIFTY24Feb2625750CE,CE,sell,2026-02-24,0,2026-02-24,2026-02-24 09:17:00,2026-02-24 09:17:00,...,27.075,58.033204,2026-02-24,2026-02-24 15:16:00,0.150000,time_exit,17.900000,9.025,18.05,25593.95
8,close,NIFTY,NIFTY24Feb2625700CE,CE,sell,2026-02-24,0,2026-02-24,2026-02-24 09:22:00,2026-02-24 09:22:00,...,24.675,58.033204,2026-02-24,2026-02-24 15:16:00,0.100000,time_exit,16.350000,8.225,16.45,25552.95
6,close,NIFTY,NIFTY24Feb2625450PE,PE,sell,2026-02-24,0,2026-02-24,2026-02-24 09:27:00,2026-02-24 09:27:00,...,31.500,16.836111,2026-02-24,2026-02-24 09:32:00,31.500000,fixed SL hit (50%),-10.500000,10.500,21.00,25567.80


In [15]:
import pandas as pd
import glob
import numpy as np
from datetime import time



START_TIME = time(9,17)
EXIT_TIME = time(15,16)
OFF_SET_TH = 0.2
QTY = 1

MAX_TRADES_CE = 2
MAX_TRADES_PE = 2

FIXED_SL_ACTIVE_TILL = time(9,34)
TRAIL_START = 0.5
TRAIL_RATIO = 0.5
TRAIL_ACTIVE_TILL = time(9,34)

ATP_TRAIL_START = time(9,35)
ATP_MULT = 1.10




In [18]:
result = []
df = pd.read_parquet('data\\nifty_2026-02-24_expiry_2026-02-24.parquet').sort_values('datetime')
pivot = (df.pivot_table(
      index = ['datetime','strike_price'],
      columns='right',
      values = 'close_opt',
      aggfunc='last'
  ).reset_index()
  .dropna(subset=['Call','Put'])
  )
pivot["straddle"] = pivot["Call"] + pivot["Put"]
expiry = pd.to_datetime(df['expiry'].iloc[0]).strftime(format='%d%b%y')
dte = ( pd.to_datetime(df['expiry'].iloc[0]) -  pd.to_datetime(df['date'].iloc[0])).days

ce_count = 0
pe_count = 0
ce_position = False
pe_position = False


In [19]:
for i,row in df.groupby('datetime'):
  if i.time() >= START_TIME:
    spot_close = row['close_spot'].iloc[0]
    atm_strike = round(spot_close/50)*50
    straddle_row = pivot[(pivot['datetime']==i) & (pivot['strike_price']==atm_strike)]
    if straddle_row.empty:
        continue
    straddle_p = straddle_row['straddle'].iloc[0]
    off_set = OFF_SET_TH*straddle_p

    call_row = (
                      row[
                          (row["right"] == "Call") &
                          (row["strike_price"] > atm_strike-50) &
                          (row["close_opt"] <= off_set)
                      ]
                      .sort_values("strike_price")
                      .head(1))

    put_row = (
                        row[
                            (row["right"] == "Put") &
                            (row["strike_price"] < atm_strike+50) &
                            (row["close_opt"] <= off_set)
                        ]
                        .sort_values("strike_price", ascending=False)
                        .head(1)
                    )
    if call_row.empty or put_row.empty:
      print(f'found empty dataframe for timestamp {i}')
      continue

    call_strike = call_row["strike_price"].iloc[0]
    put_strike  = put_row["strike_price"].iloc[0]

    call_close = call_row['close_opt'].iloc[0]
    put_close = put_row['close_opt'].iloc[0]

    call_atp = df[(df['strike_price']==call_strike) & (df['datetime']<=i) & (df['right']=='Call')]['close_opt'].mean()
    put_atp = df[(df['strike_price']==put_strike) & (df['datetime']<=i) & (df['right']=='Put')]['close_opt'].mean()
    break
    

In [20]:
call_strike

np.int64(25750)

In [21]:
call_atp

np.float64(18.116666666666664)

In [ ]:
if not ce_position :
    if call_close < call_atp and ce_count < MAX_TRADES_CE:
        ce_entry_time = i
        ce_entry_price = call_close
        ce_strike = call_strike
        ce_sl = ce_entry_price * 1.5
        ce_trailing_sl = ce_sl
        ce_entry_reason = 'Call trade occured'
        ce_spot_close = spot_close

        ce_position = True


    if not pe_position:
      if put_close < put_atp and pe_count < MAX_TRADES_PE:
        pe_entry_time = i
        pe_entry_price = put_close
        pe_strike = put_strike
        pe_sl = pe_entry_price * 1.5
        pe_trailing_sl = pe_sl
        pe_entry_reason = 'Put trade occured'
        pe_spot_close = spot_close

        pe_position = True



    if ce_position:
      ce_ltp = row[(row['strike_price']==ce_strike)&(row['right']=='Call')]['close_opt'].iloc[0]
      ce_high = row[(row['strike_price']==ce_strike)&(row['right']=='Call')]['high_opt'].iloc[0]

      atp_trail_hit = False
      if i.time() >= ATP_TRAIL_START:
          atp_trail_hit = ce_high >= (call_atp * ATP_MULT)

      profit_move = ce_entry_price - ce_ltp
      trail_hit = False
      if i.time() <= time(9,34) and profit_move >= ce_entry_price * 0.5:
          extra_profit = profit_move - (ce_entry_price * 0.5)
          trail_reduction = extra_profit * 0.5
          new_trail_sl = ce_sl - trail_reduction
          ce_trailing_sl = min(ce_trailing_sl, new_trail_sl)
          trail_hit = ce_ltp >= ce_trailing_sl

      sl_hit = ce_ltp >= ce_sl

      fixed_sl_hit = False
      if i.time()<= FIXED_SL_ACTIVE_TILL:
        fixed_sl_hit = ce_high >= (ce_sl)

      time_exit = i.time() >= EXIT_TIME

      if sl_hit or fixed_sl_hit or trail_hit or atp_trail_hit or time_exit:
        ce_exit_time = i


        if fixed_sl_hit:
            ce_exit_price = ce_sl
            ce_exit_reason = 'fixed SL hit (50%)'
        elif atp_trail_hit:
            ce_exit_price = call_atp * ATP_MULT
            ce_exit_reason = 'ATP trailing hit'
        elif trail_hit:
            ce_exit_price = ce_trailing_sl
            ce_exit_reason = 'Profit trailing SL hit'
        elif sl_hit:
            ce_exit_price = ce_sl
            ce_exit_reason = 'Initial SL hit'
        else:
            ce_exit_price = ce_ltp
            ce_exit_reason = 'time_exit'

        ce_pnl_raw = (ce_entry_price - ce_exit_price)

        result.append({
            'trade_status': 'close',
            'instrument_id': 'NIFTY',
            'symbol': f'NIFTY{expiry}{ce_strike}CE',
            'option_type': 'CE',
            'buy/sell': 'sell',
            'entry_date': ce_entry_time.date(),
            'dte': dte,
            'expiry': pd.to_datetime(expiry).strftime(format='%Y-%m-%d'),
            'entry_signal_time': ce_entry_time,
            'entry_time': ce_entry_time,
            'entry_price': ce_entry_price,
            'entry_reason': ce_entry_reason,
            'quantity': QTY,
            'market_bias': None,
            'stop_loss': ce_sl,
            'trailing_sl': call_atp * ATP_MULT,
            'exit_date': ce_exit_time.strftime(format='%Y-%m-%d'),
            'exit_signal_time': ce_exit_time,
            'exit_price': ce_exit_price,
            'exit_reason': ce_exit_reason,
            'pnl_raw': ce_pnl_raw,
            'max_loss': (ce_sl - ce_entry_price),
            'max_profit': ce_entry_price,
            'spot_close': ce_spot_close,
        })
        ce_position = False
        ce_strike = None
        ce_count+=1
    if pe_position:
      pe_ltp = row[(row['strike_price']==pe_strike)&(row['right']=='Put')]['close_opt'].iloc[0]
      pe_high = row[(row['strike_price']==pe_strike)&(row['right']=='Put')]['high_opt'].iloc[0]

      atp_trail_hit = False

      if i.time() >= ATP_TRAIL_START:
          atp_trail_hit = pe_high >= (put_atp * ATP_MULT)

      profit_move = pe_entry_price - pe_ltp
      trail_hit = False
      if i.time() <= time(9,34) and profit_move >= pe_entry_price * 0.5:
          extra_profit = profit_move - (pe_entry_price * 0.5)
          trail_reduction = extra_profit * 0.5
          new_trail_sl = pe_sl - trail_reduction
          pe_trailing_sl = min(pe_trailing_sl, new_trail_sl)
          trail_hit = pe_ltp >= pe_trailing_sl

      sl_hit = pe_ltp >= pe_sl
      fixed_sl_hit = False
      if i.time() <= FIXED_SL_ACTIVE_TILL:
        fixed_sl_hit = pe_high >= (pe_sl)

      time_exit = i.time() >= EXIT_TIME
      if sl_hit or fixed_sl_hit or trail_hit or atp_trail_hit or time_exit:
              pe_exit_time = i
              if fixed_sl_hit:
                  pe_exit_price = pe_sl
                  pe_exit_reason = 'fixed SL hit (50%)'
              elif atp_trail_hit:
                  pe_exit_price = put_atp * ATP_MULT
                  pe_exit_reason = 'ATP trailing hit'
              elif trail_hit:
                  pe_exit_price = pe_trailing_sl
                  pe_exit_reason = 'Profit trailing SL hit'
              elif sl_hit:
                  pe_exit_price = pe_sl
                  pe_exit_reason = 'Initial SL hit'
              else:
                  pe_exit_price = pe_ltp
                  pe_exit_reason = 'time_exit'

              pe_pnl_raw = (pe_entry_price - pe_exit_price)

              result.append({
                'trade_status': 'close',
                'instrument_id': 'NIFTY',
                'symbol': f'NIFTY{expiry}{pe_strike}PE',
                'option_type': 'PE',
                'buy/sell': 'sell',
                'entry_date': pe_entry_time.date(),
                'dte': dte,
                'expiry':  pd.to_datetime(expiry).strftime(format='%Y-%m-%d'),
                'entry_signal_time': pe_entry_time,
                'entry_time': pe_entry_time,
                'entry_price': pe_entry_price,
                'entry_reason': pe_exit_reason,
                'quantity': QTY,
                'market_bias': None,
                'stop_loss': pe_sl,
                'trailing_sl': put_atp * ATP_MULT,
                'exit_date': pe_exit_time.strftime(format='%Y-%m-%d'),
                'exit_signal_time': pe_exit_time,
                'exit_price': pe_exit_price,
                'exit_reason': pe_exit_reason,
                'pnl_raw': pe_pnl_raw,
                'max_loss': (pe_sl - pe_entry_price),
                'max_profit': pe_entry_price,
                'spot_close': pe_spot_close,

            })
              pe_position = False
              pe_strike = None
              pe_count+=1

In [25]:
result

[]

In [2]:
import pandas as pd
import glob
import numpy as np
from datetime import time


START_TIME = time(9,17)
EXIT_TIME = time(15,16)
OFF_SET_TH = 0.2

MAX_TRADES_CE = 2
MAX_TRADES_PE = 2

In [8]:
df = pd.read_parquet('data\\nifty_2026-02-24_expiry_2026-02-24.parquet').sort_values('datetime')
pivot = (df.pivot_table(
      index = ['datetime','strike_price'],
      columns='right',
      values = 'close_opt',
      aggfunc='last'
  ).reset_index()
  .dropna(subset=['Call','Put'])
  )
pivot["straddle"] = pivot["Call"] + pivot["Put"]

In [9]:
ce_count = 0
pe_count = 0
ce_position = False
pe_position = False
for i,row in df.groupby('datetime'):
  if i.time() >= START_TIME:
    spot_close = row['close_spot'].iloc[0]
    atm_strike = round(spot_close/50)*50
    straddle_p = pivot[(pivot['datetime']==i) & (pivot['strike_price']==atm_strike)]['straddle'].iloc[0]
    off_set = OFF_SET_TH*straddle_p
    call_row = (
                      row[
                          (row["right"] == "Call") &
                          (row["strike_price"] > atm_strike-50) &
                          (row["close_opt"] <= off_set)
                      ]
                      .sort_values("strike_price")
                      .head(1))

    put_row = (
                        row[
                            (row["right"] == "Put") &
                            (row["strike_price"] < atm_strike+50) &
                            (row["close_opt"] <= off_set)
                        ]
                        .sort_values("strike_price", ascending=False)
                        .head(1)
                    )
    if call_row.empty or put_row.empty:
      print(f'found empty dataframe for timestamp {i}')
      continue

    call_strike = call_row["strike_price"].iloc[0]
    put_strike  = put_row["strike_price"].iloc[0]

    call_close = call_row['close_opt'].iloc[0]
    put_close = put_row['close_opt'].iloc[0]

    call_atp = df[(df['strike_price']==call_strike) & (df['datetime']<=i) & (df['right']=='Call')]['close_opt'].mean()
    put_atp = df[(df['strike_price']==put_strike) & (df['datetime']<=i) & (df['right']=='Put')]['close_opt'].mean()

    if not ce_position :
      if call_close < call_atp and ce_count < MAX_TRADES_CE:
        ce_entry_time = i
        ce_entry_price = call_close
        ce_strike = call_strike
        ce_sl = ce_entry_price * 1.5

        ce_position = True
        ce_count+=1

    if not pe_position:
      if put_close < put_atp and pe_count < MAX_TRADES_PE:
        pe_entry_time = i
        pe_entry_price = put_close
        pe_strike = put_strike
        pe_sl = pe_entry_price * 1.5

        pe_position = True
        pe_count+=1


    if ce_position:
      ce_ltp = row[(row['strike_price']==ce_strike)&(row['right']=='Call')]['close_opt'].iloc[0]

      sl_hit = ce_ltp >= ce_sl
      time_exit = i.time() >= EXIT_TIME
      if sl_hit or time_exit:
        ce_exit = ce_ltp
        exit_time = i
        pnl = (ce_entry_price - ce_exit)*50
        reason = (
            'Initial sl hit' if sl_hit else
            'time'
        )

    break

In [14]:
trades = []
open_positions = []
ce_count = 0
pe_count = 0

for i,row in df.groupby('datetime'):
    if i.time() < START_TIME:
        continue
    spot_close = row['close_spot'].iloc[0]
    atm_strike = round(spot_close/50)*50
    straddle_p = pivot[(pivot['datetime']==i) & (pivot['strike_price']==atm_strike)]['straddle'].iloc[0]
    off_set = OFF_SET_TH*straddle_p
    call_row = (
                        row[
                            (row["right"] == "Call") &
                            (row["strike_price"] > atm_strike-50) &
                            (row["close_opt"] <= off_set)
                        ]
                        .sort_values("strike_price")
                        .head(1))

    put_row = (
                        row[
                            (row["right"] == "Put") &
                            (row["strike_price"] < atm_strike+50) &
                            (row["close_opt"] <= off_set)
                        ]
                        .sort_values("strike_price", ascending=False)
                        .head(1)
                    )
    if call_row.empty or put_row.empty:
        continue

    call_strike = call_row["strike_price"].iloc[0]
    put_strike  = put_row["strike_price"].iloc[0]

    call_close = call_row['close_opt'].iloc[0]
    put_close = put_row['close_opt'].iloc[0]

    call_atp = df[(df['strike_price']==call_strike) & (df['datetime']<=i) & (df['right']=='Call')]['close_opt'].mean()
    put_atp = df[(df['strike_price']==put_strike) & (df['datetime']<=i) & (df['right']=='Put')]['close_opt'].mean()

    if call_close < call_atp and ce_count < MAX_TRADES_CE:
        open_positions.append({
            'type': 'CE',
            'stike': call_strike,
            'entry_price':call_close,
            'entry_time': i
        })

        ce_count+=1

    if put_close < put_atp:
        open_positions.append({
            'type': 'PE',
            'stike': put_strike,
            'entry_price':put_close,
            'entry_time': i
        })

        pe_count+=1
    if i.time()== time(9,30):
        break


    break
